### Ex. 1 — ImageNet + Pre-trained CNN + Top-5 + Intermediate Representation


In [ ]:
!pip install -q importlib-resources tensorflow-datasets tensorflow numpy matplotlib pillow scikit-learn

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
from sklearn.model_selection import train_test_split

# 1. Load ImageNet subset: 10 classes

DATASET_DIR = "ImageNet-Mini/images"

dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="int",
    image_size=(224, 224),
    batch_size=None,
    shuffle=False
)

names = dataset.class_names

images = []
labels = []

for img, label in dataset:
    images.append(img.numpy())
    labels.append(label.numpy())

images = np.array(images)
labels = np.array(labels)

train_idx, val_idx = train_test_split(
    np.arange(len(images)),
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train = tf.data.Dataset.from_tensor_slices(
    (images[train_idx], labels[train_idx])
)

test = tf.data.Dataset.from_tensor_slices(
    (images[val_idx], labels[val_idx])
)

print("Number of classes:", len(names))
print("Training images:", len(train_idx))
print("Validation images:", len(val_idx))


# 2. Randomly select 5 test images
samples = list(test.shuffle(1000, seed=42).take(5))

imgs, labels = [], []

for img, label in samples:
    imgs.append(tf.image.resize(img, (224,224)).numpy())
    labels.append(names[int(label)])

imgs = np.array(imgs)

# 3. VGG16 preprocessing
X = preprocess_input(imgs.copy())

print("\nPreprocessing: Resize → VGG16 preprocess_input")
print("Original image:", imgs.shape)
print("Preprocessed:", X.shape)

# 4. Load pretrained VGG16
model = VGG16(weights="imagenet")
#model = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels.h5")

# Intermediate layer
feature_model = tf.keras.Model(
    model.input,
    model.get_layer("block3_conv3").output
)

# 5. CNN prediction
pred = model.predict(X, verbose=0)

# 6. Intermediate representation
features = feature_model.predict(X, verbose=0)

# 7. Display images, original labels and Top-5 predictions
for i in range(5):

    top5 = decode_predictions(pred[i:i+1], top=5)[0]

    plt.imshow(imgs[i].astype("uint8"))
    plt.title("Original: " + labels[i])
    plt.axis("off")
    plt.show()

    print("\nOriginal ImageNet class:", labels[i])
    print("Top-5 Predictions:")

    for _, name, prob in top5:
        print(f"{name:25s} {prob*100:.2f}%")

    print("Predicted class:", top5[0][1])

# 8. Visualize intermediate representation
plt.imshow(features[0,:,:,0], cmap="viridis")
plt.title("Intermediate Feature Map - block3_conv3")
plt.axis("off")
plt.show()

# 9. Output dimensions
print("\n----- OUTPUT DIMENSIONS -----")
print("Input:", X.shape)
print("Intermediate:", features.shape)
print("Final prediction:", pred.shape)

### Ex. 2 — Filters + Intermediate Feature Maps


In [ ]:
import tensorflow as tf
import the file tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array


# 1. Load one image from local ImageNet-Mini dataset

DATASET_DIR = "ImageNet-Mini/images"

dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=(224, 224),
    batch_size=None,
    shuffle=True,
    seed=42
)

names = dataset.class_names

img, label = next(iter(dataset))
x = img.numpy()
X = preprocess_input(x.copy())[None, ...]

original_label = names[int(label)]

print("Original class:", original_label)
print("Input shape:", X.shape)


# 2. Load pretrained VGG16
model = VGG16(weights="imagenet")
#model = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels.h5")

# 3. Select 3 layers: early, middle, deep
layers = ["block1_conv1", "block3_conv3", "block5_conv3"]

feature_model = tf.keras.Model(
    model.input,
    [model.get_layer(l).output for l in layers]
)

# 4. Extract feature maps
features = feature_model.predict(X, verbose=0)

# 5. Display input image
plt.imshow(x.astype("uint8"))
plt.title("Input: " + original_label)
plt.axis("off")
plt.show()

# 6. Visualize feature maps from all 3 layers
for name, f in zip(layers, features):

    print("\nLayer:", name)
    print("Feature-map dimension:", f.shape)
    plt.figure(figsize=(12,2))

    for i in range(6):
        plt.subplot(1,6,i+1)
        plt.imshow(f[0,:,:,i], cmap="viridis")
        plt.axis("off")

    plt.suptitle(name)
    plt.show()

# 7. Visualize learned filters of first convolution layer
filters = model.get_layer("block1_conv1").get_weights()[0]

print("\nFilter dimension:", filters.shape)

plt.figure(figsize=(12,2))

for i in range(6):
    plt.subplot(1,6,i+1)
    w = filters[:,:,:,i]
    w = (w-w.min())/(w.max()-w.min())
    plt.imshow(w)
    plt.axis("off")

plt.suptitle("Learned Convolutional Filters")
plt.show()

# 8. Comparison
print("\n----- LAYER COMPARISON -----")
for name, f in zip(layers, features):
    print(name, ":", f.shape)

### Ex 3 — Transfer Learning



In [ ]:
# Install beforehand, while internet is available:
# pip install tensorflow numpy matplotlib scikit-learn pillow

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *

# 1. Load local ImageNet-Mini dataset
DATASET_DIR = "ImageNet-Mini/images"

dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=(224, 224),
    batch_size=None,
    shuffle=False
)

names = dataset.class_names

images, labels = [], []

for img, label in dataset:
    images.append(img.numpy())
    labels.append(label.numpy())

images = np.array(images, dtype=np.float32)
labels = np.array(labels)

print("Number of classes:", len(names))
print("Classes:", names)
print("Total images:", len(images))

# 2. Split into 500 training and 100 test images
indices = np.arange(len(images))

train_idx, test_idx = train_test_split(
    indices,
    train_size=500,
    test_size=100,
    random_state=42,
    stratify=labels
)

Xtr_images = preprocess_input(images[train_idx].copy())
Xte_images = preprocess_input(images[test_idx].copy())

ytr = labels[train_idx]
yte = labels[test_idx]

print("Training images:", len(ytr))
print("Test images:", len(yte))

# 3. Load pretrained VGG16 using local weights
base = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",include_top=False)

extractor = Model(inputs=base.input,outputs=GlobalAveragePooling2D()(base.output))

# 4. Extract features
Xtr = extractor.predict(Xtr_images, batch_size=32, verbose=0)
Xte = extractor.predict(Xte_images, batch_size=32, verbose=0)

print("Feature vector:", Xtr.shape)

# 5. Train Logistic Regression classifier
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, ytr)

# 6. Predict test images
pred = clf.predict(Xte)

# 7. Evaluate
print("\nAccuracy :", accuracy_score(yte, pred))
prrint("Precision:", precision_score(yte, pred, average="weighted", zero_division=0))
print("Recall   :", recall_score(yte, pred, average="weighted", zero_division=0))
print("F1-score :", f1_score(yte, pred, average="weighted", zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(yte, pred))

print("\nClassification Report:")
print(classification_report(yte, pred, target_names=names, zero_division=0))

### Ex 4 - LSTM + Caption Generation

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load dataset
ds = tf.keras.utils.image_dataset_from_directory(
    "ImageNet-Mini/images", image_size=(224,224),
    batch_size=None, shuffle=True, seed=42
)
names = ds.class_names
data = list(ds.take(520))
imgs = np.array([x.numpy() for x,y in data], dtype=np.float32)
labels = np.array([y.numpy() for x,y in data])

Xtr, Xte = preprocess_input(imgs[:500]), preprocess_input(imgs[500:])
ytr, yte = labels[:500], labels[500:]

# 2. Create and tokenize captions
caps = [f"<start> a photo of a {names[int(y)]} <end>" for y in ytr]
tok = Tokenizer(filters="")
tok.fit_on_texts(caps)
seq = tok.texts_to_sequences(caps)
V, L = len(tok.word_index)+1, max(map(len,seq))

# 3. Extract VGG16 features
cnn = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
            include_top=False)
gap = GlobalAveragePooling2D()
Ftr = gap(cnn.predict(Xtr, verbose=0)).numpy()
Fte = gap(cnn.predict(Xte, verbose=0)).numpy()

# 4. Prepare next-word training data
A, B, Y = [], [], []
for i,s in enumerate(seq):
    for j in range(1,len(s)):
        A.append(Ftr[i])
        B.append(pad_sequences([s[:j]], maxlen=L-1)[0])
        Y.append(s[j])
A, B, Y = np.array(A), np.array(B), np.array(Y)

# 5. Build and train CNN-LSTM
im, wd = Input((Ftr.shape[1],)), Input((L-1,))
h = Dense(128, activation="tanh")(im)
c = Dense(128, activation="tanh")(im)
x = Embedding(V,64,mask_zero=True)(wd)
x = LSTM(128)(x, initial_state=[h,c])
model = Model([im,wd], Dense(V,activation="softmax")(x))
model.compile("adam", "sparse_categorical_crossentropy")
model.fit([A,B],Y,epochs=10,batch_size=64)

# 6. Generate captions and test
def caption(f):
    s = [tok.word_index["<start>"]]
    for _ in range(L-1):
        p = model.predict([f.reshape(1,-1),
             pad_sequences([s],maxlen=L-1)],verbose=0)[0]
        w = np.argmax(p)
        if w == tok.word_index["<end>"] or w == 0: break
        s.append(w)
    return " ".join(tok.index_word[i] for i in s[1:] if i in tok.index_word)

for i in range(min(5,len(Xte))):
    plt.imshow(imgs[500+i].astype("uint8")); plt.axis("off"); plt.show()
    print("Expected:", names[int(yte[i])])
    print("Generated:", caption(Fte[i]))

### Ex.5 - GAN + Generator + Discriminator

In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 1. Load dataset (same reference)
ds = tf.keras.utils.image_dataset_from_directory(
    "ImageNet-Mini/images", image_size=(32,32),
    batch_size=32, seed=42
)
names = ds.class_names
captions = tf.constant(["a photo of a " + n.replace("_"," ") for n in names])

# 2. Text encoder
encoder = tf.keras.layers.TextVectorization(output_mode="multi_hot")
encoder.adapt(captions)
T = encoder.vocabulary_size()

# 3. Generator: noise + text -> image
z, t = tf.keras.Input((100,)), tf.keras.Input((T,))
x = tf.keras.layers.Concatenate()([z,t])
x = tf.keras.layers.Dense(8*8*128, activation="relu")(x)
x = tf.keras.layers.Reshape((8,8,128))(x)
x = tf.keras.layers.UpSampling2D()(x)
x = tf.keras.layers.Conv2D(64,3,padding="same",activation="relu")(x)
x = tf.keras.layers.UpSampling2D()(x)
out = tf.keras.layers.Conv2D(3,3,padding="same",activation="tanh")(x)
G = tf.keras.Model([z,t],out)

# 4. Discriminator
i, c = tf.keras.Input((32,32,3)), tf.keras.Input((T,))
x = tf.keras.layers.Flatten()(i)
x = tf.keras.layers.Concatenate()([x,c])
x = tf.keras.layers.Dense(128,activation="relu")(x)
D = tf.keras.Model([i,c],tf.keras.layers.Dense(1)(x))

loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
go, do = tf.keras.optimizers.Adam(0.0002), tf.keras.optimizers.Adam(0.0002)

# 5. Adversarial training
for epoch in range(5):
    for real, labels in ds:
        real = real / 127.5 - 1
        text = encoder(tf.gather(captions, labels))
        noise = tf.random.normal((tf.shape(real)[0],100))

        with tf.GradientTape() as gt, tf.GradientTape() as dt:
            fake = G([noise,text],training=True)
            r = D([real,text],training=True)
            f = D([fake,text],training=True)
            gl = loss(tf.ones_like(f),f)
            dl = (loss(tf.ones_like(r),r)+loss(tf.zeros_like(f),f))/2

        go.apply_gradients(zip(gt.gradient(gl,G.trainable_variables),G.trainable_variables))
        do.apply_gradients(zip(dt.gradient(dl,D.trainable_variables),D.trainable_variables))

    print("Epoch",epoch+1,"G loss:",float(gl),"D loss:",float(dl))

print("GAN training completed")

# 6. Generate image for text
description = captions[0].numpy().decode()
text = encoder([description])
image = G([tf.random.normal((1,100)),text],training=False)[0].numpy()

# 7. Compare real and generated images
real, label = next(iter(ds))
plt.figure(figsize=(7,3))
plt.subplot(1,2,1)
plt.imshow(real[0].numpy().astype("uint8"))
plt.title("Real: "+names[int(label[0])])
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(np.clip((image+1)/2,0,1))
plt.title("Generated: "+description)
plt.axis("off")
plt.show()
